# Notebook ٥: Deep Learning بـ TensorFlow/Keras

شبكات عصبية على داتا FPL وعلى نوايا شات بوت الكورة. TensorFlow 2.21 وKeras 3.12، كل خلية اتشغلت فعلًا.

## ١. Tensors والطبقة الواحدة

الطبقة Dense بتعمل: y = activation(x · W + b). نحسبها بإيدينا ونقارنها بـ Keras.

In [ ]:
# المكتبات الأساسية
import os, numpy as np, tensorflow as tf
# واجهة Keras
from tensorflow import keras
# seed ثابت عشان النتايج تتكرر
keras.utils.set_random_seed(0)
# tensor شكله (1، 3)
x = tf.constant([[1.0, 2.0, 3.0]])
# الشكل والنوع
print(x.shape, x.dtype)
# طبقة بـ 2 خرج
layer = keras.layers.Dense(2, activation="relu")
# نمرر x فتتبني الأوزان
y = layer(x)
# نجيب الأوزان والـ bias
W, b = layer.get_weights()
# نحسب relu(xW+b) بإيدينا
manual = np.maximum(0, x.numpy() @ W + b)
# هل النتيجة متطابقة؟ وشكل W
print(np.allclose(y.numpy(), manual), W.shape)


## ٢. شبكة Regression على داتا FPL

نتوقع total_points من إحصائيات اللاعب، ونقارن بالـ linear regression في notebook ٤ (MAE حوالي 3.95).

In [ ]:
# جداول وقراءة JSON
import pandas as pd, json
# تقسيم الداتا
from sklearn.model_selection import train_test_split
# نقرأ الداتا
raw = json.load(open("fpl.json"))
# جدول اللاعبين
df = pd.DataFrame(raw["elements"])
# السعر الحقيقي
df["price"] = df["now_cost"] / 10
# اللي لعب 300 دقيقة فأكتر
df = df[df["minutes"] >= 300]
# الـ features
feats = ["minutes", "goals_scored", "assists", "clean_sheets", "bonus", "price"]
# مصفوفة المدخلات
X = df[feats].to_numpy("float32")
# الهدف
y = df["total_points"].to_numpy("float32")
# نفس تقسيمة notebook ٤
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
# متوسط وانحراف التدريب بس
mu, sd = Xtr.mean(0), Xtr.std(0)
# توحيد المقاييس
Xtr_s, Xte_s = (Xtr - mu) / sd, (Xte - mu) / sd
# أحجام الداتا
print(Xtr_s.shape, Xte_s.shape)


In [ ]:
# موديل طبقات متتالية
model = keras.Sequential([
# 6 مدخلات
    keras.layers.Input(shape=(6,)),
# طبقة مخفية 16 وحدة
    keras.layers.Dense(16, activation="relu"),
# طبقة مخفية 8 وحدات
    keras.layers.Dense(8, activation="relu"),
# خرج واحد: النقاط
    keras.layers.Dense(1),
# نقفل الموديل
])
# نحدد المحسّن ودالة الخسارة
model.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse", metrics=["mae"])
# نتدرب 60 دورة
hist = model.fit(Xtr_s, ytr, epochs=60, batch_size=32, validation_split=0.15, verbose=0)
# دورات مختارة
for e in [0, 9, 19, 39, 59]:
# نطبع الخسارة وخطأ التحقق
    print(f"epoch {e+1:2}: loss={hist.history['loss'][e]:7.2f}  val_mae={hist.history['val_mae'][e]:5.2f}")
# الخطأ على الاختبار
mae = model.evaluate(Xte_s, yte, verbose=0)[1]
# نطبعه
print("test MAE:", round(float(mae), 2))


الملاحظة: خطأ الاختبار هنا أكبر من الـ linear regression في notebook ٤ (3.95). الشبكة مش دايمًا أحسن على داتا جدولية صغيرة، وده طبيعي.

## ٣. شبكة نصوص: نية المستخدم

TextVectorization يحول النص لأرقام، وEmbedding يحول كل رقم لمتجه، والـ pooling يلخص الجملة.

In [ ]:
# جمل ونوايا (من كتابتنا)
data = {
# إحصائيات
 "stats": ["how many goals did salah score", "show me haaland stats", "assists for rice this season", "top scorer in the league", "how many minutes did saka play", "give me player points", "who has the most assists", "goals and assists of palmer", "what are the stats for son", "clean sheets for raya", "points per game of watkins", "best player this season"],
# إصابات
 "injury": ["is salah injured", "any injury news for saka", "is haaland fit to play", "who is out with injury", "when will rice return", "ankle injury update", "is he suspended", "knock on the hamstring", "fitness news for the arsenal squad", "is palmer doubtful", "who is ruled out this week", "injured players list"],
# انتقالات
 "transfer": ["did he join a new club", "any transfer news", "who left on loan", "is salah leaving liverpool", "new signing at arsenal", "loan deal for the winger", "who joined chelsea", "player sold this summer", "transfer window rumours", "has he signed a contract", "who moved on loan", "latest signings"],
# مباريات
 "fixtures": ["when is the next match", "who do arsenal play next", "fixtures for this week", "what time is the game", "next gameweek schedule", "when does liverpool play", "show upcoming fixtures", "is there a match tomorrow", "arsenal vs chelsea date", "kick off time today", "next opponent for city", "match calendar"],
# تحيات
 "greeting": ["hello", "hi there", "good morning", "hey bot", "hello how are you", "hi", "good evening", "hey there", "hello again", "morning", "hi friend", "hey"],
# نقفل القاموس
}
# أسماء النوايا مرتبة
labels = sorted(data)
# كل الجمل
texts = [t for k in labels for t in data[k]]
# رقم النية لكل جملة
ids = np.array([labels.index(k) for k in labels for _ in data[k]])
# تقسيم تدريب واختبار
Ttr, Tte, ltr, lte = train_test_split(texts, ids, test_size=0.25, random_state=0, stratify=ids)
# الأعداد وأسماء النوايا
print(len(Ttr), len(Tte), labels)


In [ ]:
# محوّل نص لأرقام بطول 8
vec = keras.layers.TextVectorization(max_tokens=200, output_sequence_length=8)
# يبني الـ vocabulary من التدريب
vec.adapt(Ttr)
# أول 10 كلمات في الـ vocab
print([str(w) for w in vec.get_vocabulary()[:10]])
# جملة متحولة لأرقام (الصفر حشو)
print(vec(["is salah injured"]).numpy())


In [ ]:
# seed ثابت
keras.utils.set_random_seed(0)
# موديل النية
nlp = keras.Sequential([
# المدخل: نص واحد لكل صف
    keras.layers.Input(shape=(), dtype="string"),
# تحويل النص لأرقام
    vec,
# كل كلمة لمتجه 16 رقم
    keras.layers.Embedding(input_dim=vec.vocabulary_size(), output_dim=16),
# متوسط متجهات الكلمات = متجه الجملة
    keras.layers.GlobalAveragePooling1D(),
# نطفي 20% عشوائي لتقليل الـ overfitting
    keras.layers.Dropout(0.2),
# احتمال كل نية
    keras.layers.Dense(len(labels), activation="softmax"),
# نقفل الموديل
])
# تجهيز التدريب
nlp.compile(optimizer=keras.optimizers.Adam(0.02), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
# نلف على الطبقات
for l in nlp.layers:
# اسم الطبقة وعدد أوزانها
    print(f"{l.name:28} params={l.count_params():>5}")
# إجمالي الأوزان
print("total:", nlp.count_params())


In [ ]:
# تدريب 60 دورة
h = nlp.fit(tf.constant(Ttr), ltr, epochs=60, batch_size=8, verbose=0)
# دورات مختارة
for e in [0, 19, 39, 59]:
# الخسارة والدقة
    print(f"epoch {e+1:2}: loss={h.history['loss'][e]:.3f} acc={h.history['accuracy'][e]:.2f}")
# تقييم على الاختبار
loss, acc = nlp.evaluate(tf.constant(Tte), lte, verbose=0)
# دقة الاختبار
print("test accuracy:", round(float(acc), 2))


الملاحظة: دقة التدريب 1.00 ودقة الاختبار 0.53. ده overfitting واضح، لأن عندنا 45 جملة تدريب بس. الحل الحقيقي داتا أكتر أو embeddings جاهزة (BERT)، وده اللي بنوصله في الـ notebooks الجاية.

In [ ]:
# دالة تسأل البوت
def ask(q):
# احتمالات النوايا
    p = nlp.predict(tf.constant([q]), verbose=0)[0]
# النية والثقة
    return labels[int(p.argmax())], round(float(p.max()), 2)
# أسئلة تجريبية
for q in ["is rice injured", "when do arsenal play", "hello bot", "salah assists this season"]:
# نطبع السؤال والنتيجة
    print(q, "->", ask(q))


## ٤. Embeddings اللي الموديل اتعلمها

كل كلمة بقت متجه. نقيس تشابه كلمتين بالـ cosine زي فصل الـ embeddings.

In [ ]:
# مصفوفة الـ embedding (كلمات × 16)
emb = nlp.layers[1].get_weights()[0]
# قايمة الكلمات
vocab = [str(w) for w in vec.get_vocabulary()]
# شكل المصفوفة
print(emb.shape)
# دالة cosine
def cos(a, b):
# ضرب داخلي على الأطوال
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
# أزواج للمقارنة
for w1, w2 in [("injured", "injury"), ("injured", "hello"), ("match", "fixtures")]:
# لو الكلمتين في الـ vocab
    if w1 in vocab and w2 in vocab:
# نطبع التشابه
        print(w1, w2, round(cos(emb[vocab.index(w1)], emb[vocab.index(w2)]), 2))


## ٥. حفظ وتحميل وEarlyStopping

نحفظ الموديل عشان Streamlit، ونوقف التدريب لما التحسن يقف.

In [ ]:
# نحفظ الموديل كله
nlp.save("intent_nlp.keras")
# نحمله تاني
again = keras.models.load_model("intent_nlp.keras")
# سؤال تجريبي
q = tf.constant(["any injury news for rice"])
# النية اللي توقعها الموديل المحمّل
print(labels[int(again.predict(q, verbose=0)[0].argmax())])
# نوقف لو val_loss ما تحسنش 5 دورات
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
# نسخة جديدة بنفس تصميم موديل FPL
m2 = keras.models.clone_model(model)
# نجهزها
m2.compile(optimizer="adam", loss="mse")
# نتدرب مع الإيقاف المبكر
h2 = m2.fit(Xtr_s, ytr, epochs=300, validation_split=0.2, callbacks=[es], verbose=0)
# عدد الدورات الفعلي
print("stopped after", len(h2.history["loss"]), "epochs")


الملاحظة: الـ EarlyStopping ما أوقفش التدريب قبل 300 دورة في الجولة دي (val_loss فضل بيتحسن)، فاتسجل العدد الحقيقي زي ما طلع.